# Reto: Análisis de deserción de empleados con sus sistemas operativos utilizados

**Curso:** Manipulación de datos con Python · The Learning Gate | Tecnológico de Monterrey
**Entregable:** `DS_C2_SC3_CarlosAlbertoTapiaHernandez` (libreta Colab + archivo `.xlsx` con los resultados)

---

## Contexto
Se trabaja con los datos del *Hackathon HackerEarth 2020* (información de empleados) junto con
información del departamento de TI sobre los equipos que usa cada empleado. Los datos vienen en
**cuatro archivos**:

| Archivo | Contenido | Registros |
|---|---|---|
| `Train.csv` | Datos de empleados (edad, nivel educativo, estado civil, …) + `Attrition_rate` | 7,000 |
| `Test.csv` | Datos de empleados (mismas columnas, **sin** `Attrition_rate`) | 3,000 |
| `Train2.csv` | Datos de TI: SO de escritorio y móvil, versiones y tickets | 7,000 |
| `Test2.csv` | Datos de TI (misma estructura que `Train2.csv`) | 3,000 |

## Plan de trabajo
1. Cargar los cuatro archivos en DataFrames y explorarlos.
2. **Concatenar** `Train` + `Test` (empleados) y `Train2` + `Test2` (TI) → dos DataFrames con estructura común.
3. **Relacionar** ambos DataFrames con `Employee_ID` (`merge`).
4. Reemplazar los sistemas operativos vacíos por `"No registrado"`.
5. Resolver las cinco consultas solicitadas.
6. Exportar cada consulta a su propia hoja de un archivo `.xlsx`.

## 0. Configuración

La variable `NOMBRE` se usa para nombrar el archivo de salida como pide el reto:
`DS_C2_SC3_CarlosAlbertoTapiaHernandez.xlsx`.

In [1]:
import glob

import pandas as pd

NOMBRE = "CarlosAlbertoTapiaHernandez"
ARCHIVO_XLSX = f"DS_C2_SC3_{NOMBRE}.xlsx"

print("pandas", pd.__version__)

pandas 3.1.1


## 1. Carga de los archivos en DataFrames

### 1.1 Subir los archivos a Colab
Si los cuatro `.csv` **no** están en el directorio de trabajo (por ejemplo, la primera vez que abres esta
libreta en Colab), la celda siguiente abre el selector de archivos para que los subas
(`Train.csv`, `Test.csv`, `Train2.csv` y `Test2.csv`). Si ya están en la carpeta, no hace nada.

> La función `buscar_archivo` localiza cada archivo por su terminación, así que también funciona si
> la descarga les agregó un prefijo al nombre (p. ej. `0.abc123Train.csv`).

In [2]:
ARCHIVOS = ["Train.csv", "Test.csv", "Train2.csv", "Test2.csv"]


def buscar_archivo(nombre):
    """Devuelve la ruta del archivo cuyo nombre termina en `nombre`, o None si no existe."""
    coincidencias = sorted(glob.glob(f"*{nombre}"))
    return coincidencias[0] if coincidencias else None


faltantes = [a for a in ARCHIVOS if buscar_archivo(a) is None]

if faltantes:
    try:
        from google.colab import files          # solo existe dentro de Google Colab
        print("Selecciona los archivos que faltan:", faltantes)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Faltan los archivos {faltantes} en la carpeta de trabajo.")

rutas = {a: buscar_archivo(a) for a in ARCHIVOS}
rutas

{'Train.csv': 'Train.csv',
 'Test.csv': 'Test.csv',
 'Train2.csv': 'Train2.csv',
 'Test2.csv': 'Test2.csv'}

### 1.2 Crear los cuatro DataFrames

Al abrir los archivos se detectó un detalle: en `Train2.csv` y `Test2.csv` los encabezados vienen con un
**espacio al inicio** (`", Computer_OS"`, `", Mobile_OS"`, …). Si no se corrige, `df["Computer_OS"]`
produciría un `KeyError`. Con `skipinitialspace=True` pandas ignora esos espacios al leer.

In [3]:
df_train  = pd.read_csv(rutas["Train.csv"])    # empleados (entrenamiento)
df_test   = pd.read_csv(rutas["Test.csv"])     # empleados (prueba)

# Los archivos de TI traen espacios después de las comas en el encabezado
df_train2 = pd.read_csv(rutas["Train2.csv"], skipinitialspace=True)   # TI (entrenamiento)
df_test2  = pd.read_csv(rutas["Test2.csv"],  skipinitialspace=True)   # TI (prueba)

for nombre, df in [("df_train", df_train), ("df_test", df_test),
                   ("df_train2", df_train2), ("df_test2", df_test2)]:
    print(f"{nombre:10s} -> {df.shape[0]:>5,} filas x {df.shape[1]:>2} columnas")

df_train   -> 7,000 filas x 24 columnas
df_test    -> 3,000 filas x 23 columnas
df_train2  -> 7,000 filas x  7 columnas
df_test2   -> 3,000 filas x  7 columnas


### 1.3 Exploración de la estructura

In [4]:
print("Columnas de df_train :", df_train.columns.tolist())
print()
print("Columnas de df_test  :", df_test.columns.tolist())
print()
print("Columnas de df_train2:", df_train2.columns.tolist())
print("Columnas de df_test2 :", df_test2.columns.tolist())

Columnas de df_train : ['Employee_ID', 'Gender', 'Age', 'Education_Level', 'Relationship_Status', 'Hometown', 'Unit', 'Decision_skill_possess', 'Time_of_service', 'Time_since_promotion', 'growth_rate', 'Travel_Rate', 'Post_Level', 'Pay_Scale', 'Compensation_and_Benefits', 'Work_Life_balance', 'VAR1', 'VAR2', 'VAR3', 'VAR4', 'VAR5', 'VAR6', 'VAR7', 'Attrition_rate']

Columnas de df_test  : ['Employee_ID', 'Gender', 'Age', 'Education_Level', 'Relationship_Status', 'Hometown', 'Unit', 'Decision_skill_possess', 'Time_of_service', 'Time_since_promotion', 'growth_rate', 'Travel_Rate', 'Post_Level', 'Pay_Scale', 'Compensation_and_Benefits', 'Work_Life_balance', 'VAR1', 'VAR2', 'VAR3', 'VAR4', 'VAR5', 'VAR6', 'VAR7']

Columnas de df_train2: ['Employee_ID', 'Computer_OS', 'Computer_OS_version', 'Computer_tickets', 'Mobile_OS', 'Mobile_OS_version', 'Mobile_tickets']
Columnas de df_test2 : ['Employee_ID', 'Computer_OS', 'Computer_OS_version', 'Computer_tickets', 'Mobile_OS', 'Mobile_OS_version', 

In [5]:
df_train.head()

,Employee_ID,Gender,Age,Education_Level,Relationship_Status,Hometown,Unit,Decision_skill_possess,Time_of_service,Time_since_promotion,...,Compensation_and_Benefits,Work_Life_balance,VAR1,VAR2,VAR3,VAR4,VAR5,VAR6,VAR7,Attrition_rate
0,EID_23371,F,42.0,4,Married,Franklin,IT,Conceptual,4.0,4,...,type2,3.0,4,0.7516,1.8688,2.0,4,5,3,0.1841
1,EID_18000,M,24.0,3,Single,Springfield,Logistics,Analytical,5.0,4,...,type2,4.0,3,-0.9612,-0.4537,2.0,3,5,3,0.0670
2,EID_3891,F,58.0,3,Married,Clinton,Quality,Conceptual,27.0,3,...,type2,1.0,4,-0.9612,-0.4537,3.0,3,8,3,0.0851
3,EID_17492,F,26.0,3,Single,Lebanon,Human Resource Management,Behavioral,4.0,3,...,type2,1.0,3,-1.8176,-0.4537,NaN,3,7,3,0.0668
4,EID_22534,F,31.0,1,Married,Springfield,Logistics,Conceptual,5.0,4,...,type3,3.0,1,0.7516,-0.4537,2.0,2,8,2,0.1827


In [6]:
df_train2.head()

,Employee_ID,Computer_OS,Computer_OS_version,Computer_tickets,Mobile_OS,Mobile_OS_version,Mobile_tickets
0,EID_18343,Windows,8,1,Android,9.0,0
1,EID_4861,Windows,7,2,Android,8.0,0
2,EID_16936,Windows,10,3,Android,10.0,0
3,EID_16384,Windows,10,1,Android,7.0,1
4,EID_10666,Windows,7,7,Android,10.0,4


Revisamos tipos de dato, valores nulos y duplicados en la llave `Employee_ID`, que será la columna de
relación entre los archivos.

In [7]:
resumen = pd.DataFrame({
    "filas":               [len(d) for d in (df_train, df_test, df_train2, df_test2)],
    "Employee_ID nulos":   [d["Employee_ID"].isna().sum() for d in (df_train, df_test, df_train2, df_test2)],
    "Employee_ID repetidos": [d["Employee_ID"].duplicated().sum() for d in (df_train, df_test, df_train2, df_test2)],
}, index=["df_train", "df_test", "df_train2", "df_test2"])
resumen

,filas,Employee_ID nulos,Employee_ID repetidos
df_train,7000,0,0
df_test,3000,0,0
df_train2,7000,0,0
df_test2,3000,0,0


In [8]:
# Nulos por columna en los archivos de TI (las que nos interesan para las consultas)
pd.concat([df_train2, df_test2]).isna().sum()

Employee_ID              0
Computer_OS            158
Computer_OS_version    360
Computer_tickets         0
Mobile_OS              287
Mobile_OS_version      319
Mobile_tickets           0
dtype: int64

## 2. Concatenar: dos DataFrames con estructura común

`Train` y `Test` tienen las mismas columnas de empleados (la única diferencia es que `Test` no incluye
`Attrition_rate`), y `Train2` y `Test2` tienen exactamente la misma estructura de TI.
Por eso se **apilan** con `pd.concat`:

* `df_empleados` = `Train` + `Test`  →  10,000 empleados
* `df_ti`        = `Train2` + `Test2` →  10,000 registros de TI

Con `ignore_index=True` se reconstruye el índice (0 … 9,999). Los 3,000 empleados de `Test` quedan con
`Attrition_rate` vacío (`NaN`), lo cual es lo esperado: ese dato no existe en el archivo de prueba.

In [9]:
df_empleados = pd.concat([df_train, df_test],   ignore_index=True)
df_ti        = pd.concat([df_train2, df_test2], ignore_index=True)

print("df_empleados:", df_empleados.shape)
print("df_ti       :", df_ti.shape)
print()
print("¿Employee_ID únicos en df_empleados?", df_empleados["Employee_ID"].is_unique)
print("¿Employee_ID únicos en df_ti?       ", df_ti["Employee_ID"].is_unique)
print("Attrition_rate vacío (filas de Test):", df_empleados["Attrition_rate"].isna().sum())

df_empleados: (10000, 24)
df_ti       : (10000, 7)

¿Employee_ID únicos en df_empleados? True
¿Employee_ID únicos en df_ti?        True
Attrition_rate vacío (filas de Test): 3000


## 3. Relacionar la información con `Employee_ID`

Se unen los datos del empleado con los datos de TI mediante `merge` usando `Employee_ID` como llave.

* `how="left"`: se conservan **todos** los empleados aunque no tuvieran registro de TI.
* `validate="one_to_one"`: pandas lanza un error si algún `Employee_ID` estuviera repetido en alguno de los
  dos lados (garantiza que el `merge` no duplique filas).
* `indicator=True`: agrega la columna `_merge` para comprobar que cada empleado encontró su pareja.

In [10]:
df = df_empleados.merge(df_ti, on="Employee_ID", how="left",
                        validate="one_to_one", indicator=True)

print("Resultado del merge:")
print(df["_merge"].value_counts().to_string())

# Validaciones: 10,000 filas, todas con pareja, y ningún ID de TI quedó sin empleado
assert len(df) == 10_000
assert (df["_merge"] == "both").all()
assert set(df_ti["Employee_ID"]) == set(df_empleados["Employee_ID"])

df = df.drop(columns="_merge")
print()
print("DataFrame final:", df.shape)

Resultado del merge:
_merge
both          10000
left_only         0
right_only        0

DataFrame final: (10000, 30)


In [11]:
df.head()

,Employee_ID,Gender,Age,Education_Level,Relationship_Status,Hometown,Unit,Decision_skill_possess,Time_of_service,Time_since_promotion,...,VAR5,VAR6,VAR7,Attrition_rate,Computer_OS,Computer_OS_version,Computer_tickets,Mobile_OS,Mobile_OS_version,Mobile_tickets
0,EID_23371,F,42.0,4,Married,Franklin,IT,Conceptual,4.0,4,...,4,5,3,0.1841,Windows,8,1,Android,7.0,0
1,EID_18000,M,24.0,3,Single,Springfield,Logistics,Analytical,5.0,4,...,3,5,3,0.0670,MacOS,Mojave,1,iOS,13.0,1
2,EID_3891,F,58.0,3,Married,Clinton,Quality,Conceptual,27.0,3,...,3,8,3,0.0851,Windows,8,1,Android,7.0,0
3,EID_17492,F,26.0,3,Single,Lebanon,Human Resource Management,Behavioral,4.0,3,...,3,7,3,0.0668,Windows,7,1,Android,8.0,0
4,EID_22534,F,31.0,1,Married,Springfield,Logistics,Conceptual,5.0,4,...,2,8,2,0.1827,Windows,10,9,Android,8.0,1


## 4. Sistemas operativos sin valor → `"No registrado"`

Hay empleados sin sistema operativo registrado, tanto de escritorio (`Computer_OS`) como móvil
(`Mobile_OS`). Según el reto, a todos ellos se les asigna el valor `"No registrado"`.

In [12]:
print("Antes:")
print("  Computer_OS vacíos:", df["Computer_OS"].isna().sum())
print("  Mobile_OS vacíos  :", df["Mobile_OS"].isna().sum())

df[["Computer_OS", "Mobile_OS"]] = df[["Computer_OS", "Mobile_OS"]].fillna("No registrado")

print()
print("Después:")
print("  Computer_OS vacíos:", df["Computer_OS"].isna().sum())
print("  Mobile_OS vacíos  :", df["Mobile_OS"].isna().sum())

Antes:
  Computer_OS vacíos: 158
  Mobile_OS vacíos  : 287

Después:
  Computer_OS vacíos: 0
  Mobile_OS vacíos  : 0


In [13]:
print(df["Computer_OS"].value_counts().to_string())
print()
print(df["Mobile_OS"].value_counts().to_string())

Computer_OS
Windows          7728
MacOS            1634
Linux             480
No registrado     158

Mobile_OS
Android          7659
iOS              2054
No registrado     287


## 5. Consultas

Para que las tablas se lean siempre en el mismo orden se definen las categorías de cada SO.

> **Nota:** en los datos el sistema operativo de Apple en escritorio aparece como **`MacOS`**
> (el enunciado lo llama "Mac"); se respeta el valor tal como viene en la base.

In [14]:
ORDEN_ESCRITORIO = ["Windows", "MacOS", "Linux", "No registrado"]
ORDEN_MOVIL      = ["Android", "iOS", "No registrado"]

### Consulta 1 · Porcentaje de personas por SO de escritorio
`value_counts(normalize=True)` devuelve la proporción de cada categoría; se multiplica por 100.
**Resultado esperado: 4 renglones.**

In [15]:
consulta1 = (df["Computer_OS"]
             .value_counts(normalize=True)
             .mul(100).round(2)
             .rename("Porcentaje")
             .rename_axis("Sistema_Operativo_Escritorio")
             .reset_index())

print("Suma de porcentajes:", consulta1["Porcentaje"].sum().round(2), "| renglones:", len(consulta1))
consulta1

Suma de porcentajes: 100.0 | renglones: 4


,Sistema_Operativo_Escritorio,Porcentaje
0,Windows,77.28
1,MacOS,16.34
2,Linux,4.80
3,No registrado,1.58


### Consulta 2 · Porcentaje de personas por SO móvil
Misma lógica sobre `Mobile_OS`. **Resultado esperado: 3 renglones.**

In [16]:
consulta2 = (df["Mobile_OS"]
             .value_counts(normalize=True)
             .mul(100).round(2)
             .rename("Porcentaje")
             .rename_axis("Sistema_Operativo_Movil")
             .reset_index())

print("Suma de porcentajes:", consulta2["Porcentaje"].sum().round(2), "| renglones:", len(consulta2))
consulta2

Suma de porcentajes: 100.0 | renglones: 3


,Sistema_Operativo_Movil,Porcentaje
0,Android,76.59
1,iOS,20.54
2,No registrado,2.87


### Consulta 3 · Edad promedio por SO de escritorio
`groupby("Computer_OS")["Age"].mean()`. La columna `Age` tiene valores vacíos (573 empleados); `mean()`
los **ignora** por defecto, de modo que el promedio se calcula con las edades disponibles.

In [17]:
consulta3 = (df.groupby("Computer_OS")["Age"]
             .mean().round(2)
             .reindex(ORDEN_ESCRITORIO)
             .rename("Edad_Promedio")
             .rename_axis("Sistema_Operativo_Escritorio")
             .reset_index())
consulta3

,Sistema_Operativo_Escritorio,Edad_Promedio
0,Windows,39.46
1,MacOS,40.40
2,Linux,40.45
3,No registrado,40.52


### Consulta 4 · Nivel educativo promedio por SO de escritorio
`Education_Level` es una escala numérica de 1 a 5, por lo que se puede promediar directamente.

In [18]:
consulta4 = (df.groupby("Computer_OS")["Education_Level"]
             .mean().round(2)
             .reindex(ORDEN_ESCRITORIO)
             .rename("Nivel_Educativo_Promedio")
             .rename_axis("Sistema_Operativo_Escritorio")
             .reset_index())
consulta4

,Sistema_Operativo_Escritorio,Nivel_Educativo_Promedio
0,Windows,3.20
1,MacOS,3.17
2,Linux,3.24
3,No registrado,3.18


### Consulta 5 · Número total de tickets por SO de escritorio
Se **suman** los tickets de escritorio (`Computer_tickets`) de todas las personas de cada SO.

In [19]:
consulta5 = (df.groupby("Computer_OS")["Computer_tickets"]
             .sum()
             .reindex(ORDEN_ESCRITORIO)
             .rename("Total_Tickets")
             .rename_axis("Sistema_Operativo_Escritorio")
             .reset_index())

print("Total de tickets de escritorio:", consulta5["Total_Tickets"].sum(),
      "| coincide con la columna completa:", df["Computer_tickets"].sum())
consulta5

Total de tickets de escritorio: 23844 | coincide con la columna completa: 23844


,Sistema_Operativo_Escritorio,Total_Tickets
0,Windows,21169
1,MacOS,1880
2,Linux,254
3,No registrado,541


## 6. Exportar los resultados a Excel

Cada consulta se guarda en **su propia hoja** de un mismo archivo `.xlsx`
(`DS_C2_SC3_CarlosAlbertoTapiaHernandez.xlsx`) usando `pd.ExcelWriter`. Además se ajusta el ancho de las columnas para que
sea legible al abrirlo.

In [20]:
hojas = {
    "1_Pct_SO_Escritorio":    consulta1,
    "2_Pct_SO_Movil":         consulta2,
    "3_Edad_Promedio":        consulta3,
    "4_Nivel_Educativo_Prom": consulta4,
    "5_Tickets_Totales":      consulta5,
}

with pd.ExcelWriter(ARCHIVO_XLSX, engine="openpyxl") as writer:
    for nombre_hoja, tabla in hojas.items():
        tabla.to_excel(writer, sheet_name=nombre_hoja, index=False)
        ws = writer.sheets[nombre_hoja]
        for col in ws.columns:                       # ancho de columna = texto más largo
            ancho = max(len(str(c.value)) for c in col) + 3
            ws.column_dimensions[col[0].column_letter].width = ancho

print("Archivo generado:", ARCHIVO_XLSX)

Archivo generado: DS_C2_SC3_CarlosAlbertoTapiaHernandez.xlsx


### Verificación: se vuelve a leer el `.xlsx` para comprobar el contenido de cada hoja

In [21]:
hojas_leidas = pd.read_excel(ARCHIVO_XLSX, sheet_name=None)   # dict {nombre_hoja: DataFrame}
print("Hojas en el archivo:", list(hojas_leidas), "\n")

for hoja, tabla in hojas_leidas.items():
    print(f"=== {hoja} ===")
    print(tabla.to_string(index=False), "\n")

Hojas en el archivo: ['1_Pct_SO_Escritorio', '2_Pct_SO_Movil', '3_Edad_Promedio', '4_Nivel_Educativo_Prom', '5_Tickets_Totales'] 

=== 1_Pct_SO_Escritorio ===
Sistema_Operativo_Escritorio  Porcentaje
                     Windows       77.28
                       MacOS       16.34
                       Linux        4.80
               No registrado        1.58 

=== 2_Pct_SO_Movil ===
Sistema_Operativo_Movil  Porcentaje
                Android       76.59
                    iOS       20.54
          No registrado        2.87 

=== 3_Edad_Promedio ===
Sistema_Operativo_Escritorio  Edad_Promedio
                     Windows          39.46
                       MacOS          40.40
                       Linux          40.45
               No registrado          40.52 

=== 4_Nivel_Educativo_Prom ===
Sistema_Operativo_Escritorio  Nivel_Educativo_Promedio
                     Windows                      3.20
                       MacOS                      3.17
                       

### Descargar el archivo
En Google Colab, la siguiente celda descarga el `.xlsx` a tu computadora (fuera de Colab no hace nada).

In [22]:
try:
    from google.colab import files
    files.download(ARCHIVO_XLSX)
except ImportError:
    print("Fuera de Colab: el archivo ya quedó guardado en la carpeta de trabajo ->", ARCHIVO_XLSX)

Fuera de Colab: el archivo ya quedó guardado en la carpeta de trabajo -> DS_C2_SC3_CarlosAlbertoTapiaHernandez.xlsx


## 7. Conclusiones

* **SO de escritorio:** Windows domina con **77.28 %** de las personas; le siguen MacOS (16.34 %) y
  Linux (4.80 %). Solo el **1.58 %** no tiene SO de escritorio registrado.
* **SO móvil:** Android lidera con **76.59 %**, iOS tiene 20.54 % y el 2.87 % no está registrado.
* **Edad:** quienes usan Windows son ligeramente más jóvenes (**39.46 años**) que quienes usan MacOS (40.40),
  Linux (40.45) o no tienen registro (40.52). La diferencia es de apenas un año.
* **Nivel educativo:** es prácticamente igual en todos los grupos (entre **3.17 y 3.24** en la escala 1-5),
  así que el SO de escritorio no distingue el nivel educativo.
* **Tickets:** Windows concentra **21,169** de los 23,844 tickets de escritorio (88.8 %). Esto es esperable
  porque también es el SO con más usuarios; para comparar la carga de soporte entre sistemas convendría
  analizar los **tickets por persona**, no solo el total.